# 01 Fetch & Chunk

AAPL/MSFT/GOOGL の 10-K × 5年 + 10-Q × 15四半期 = 60 件を取得し、
Item 1A (Risk Factors) と Item 7/Item 2 (MD&A) を抽出、
FinBERT トークナイザで 510 トークンチャンクに分割する。

In [ ]:
# Cell 1: imports + setup (必ず最初に _helpers を import)
# Jupyter のモジュールキャッシュ対策: 古い _helpers / edgar を退避してから再 import
import sys
from pathlib import Path
for _m in [m for m in list(sys.modules) if m == '_helpers' or m == 'edgar' or m.startswith('edgar.')]:
    del sys.modules[_m]
sys.path.insert(0, str(Path.cwd()))
import _helpers
_ = _helpers.setup_edgar()
device = _helpers.get_device()
print('device:', device)
print('DATA_DIR:', _helpers.DATA_DIR)


In [ ]:
# Cell 2: 10-K を 3 銘柄 × 5 年取得 (edgartools 直接利用)
import edgar
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

TICKERS = ['AAPL', 'MSFT', 'GOOGL']

def _fetch(ticker, form, limit):
    try:
        return ticker, list(edgar.Company(ticker).get_filings(form=form).head(limit))
    except Exception as e:  # noqa: BLE001
        return ticker, e

with ThreadPoolExecutor(max_workers=3) as ex:
    filings_10k = dict(ex.map(lambda t: _fetch(t, '10-K', 5), TICKERS))
for t, r in filings_10k.items():
    print(t, len(r) if not isinstance(r, Exception) else f'ERR: {r}')


In [ ]:
# Cell 3: 10-Q を 3 銘柄 × 15 四半期取得
with ThreadPoolExecutor(max_workers=3) as ex:
    filings_10q = dict(ex.map(lambda t: _fetch(t, '10-Q', 15), TICKERS))
for t, r in filings_10q.items():
    print(t, len(r) if not isinstance(r, Exception) else f'ERR: {r}')


In [ ]:
# Cell 4: filings メタを 1 つの DataFrame に統合し filings.parquet 保存
import pandas as pd

all_filing_objs = []
for d, label in [(filings_10k, '10-K'), (filings_10q, '10-Q')]:
    for ticker, result in d.items():
        if isinstance(result, Exception):
            continue
        for f in result:
            all_filing_objs.append((ticker, label, f))

df_filings = pd.DataFrame([
    {
        'filing_id': str(f.accession_number),
        'ticker': ticker,
        'form': form,
        'filing_date': pd.Timestamp(str(f.filing_date)),
        'accession_number': str(f.accession_number),
    }
    for ticker, form, f in all_filing_objs
])
df_filings = df_filings.sort_values(['ticker', 'form', 'filing_date']).reset_index(drop=True)
df_filings.to_parquet(_helpers.FILINGS_PARQUET)
print('saved:', _helpers.FILINGS_PARQUET, 'rows:', len(df_filings))
df_filings.head()


In [ ]:
# Cell 5: 10-K のセクション抽出 (edgartools TenK ネイティブ accessor)
# TenK.risk_factors -> Item 1A、 TenK.management_discussion -> Item 7
section_rows = []
miss = []
for ticker, form, f in tqdm([x for x in all_filing_objs if x[1] == '10-K'], desc='10-K sections'):
    try:
        obj = f.obj()  # TenK インスタンス
    except Exception as e:  # noqa: BLE001
        print(f'10-K obj fail: {ticker} {f.accession_number} {e}')
        continue
    fid = str(f.accession_number)
    for key, attr in [('item_1a', 'risk_factors'), ('item_7', 'management_discussion')]:
        text = getattr(obj, attr, None)
        if isinstance(text, str) and text:
            section_rows.append({
                'filing_id': fid, 'section_key': key,
                'text': text, 'char_count': len(text),
            })
        else:
            miss.append((ticker, fid, '10-K', key))
print('10-K sections extracted:', len(section_rows), 'miss:', len(miss))


In [ ]:
# Cell 6: 10-Q のセクション抽出 (edgartools TenQ ネイティブ subscript)
# obj['Part II, Item 1A'] -> Risk Factors、 obj['Part I, Item 2'] -> MD&A
for ticker, form, f in tqdm([x for x in all_filing_objs if x[1] == '10-Q'], desc='10-Q sections'):
    try:
        obj = f.obj()  # TenQ インスタンス
    except Exception as e:  # noqa: BLE001
        print(f'10-Q obj fail: {ticker} {f.accession_number} {e}')
        continue
    fid = str(f.accession_number)
    for key, idx in [('item_1a', 'Part II, Item 1A'), ('item_7', 'Part I, Item 2')]:
        try:
            text = obj[idx]
        except (KeyError, TypeError):
            text = None
        if isinstance(text, str) and text:
            section_rows.append({
                'filing_id': fid, 'section_key': key,
                'text': text, 'char_count': len(text),
            })
        else:
            miss.append((ticker, fid, '10-Q', key))
print('total sections:', len(section_rows), 'total miss:', len(miss))


In [ ]:
# Cell 7: sections.parquet 保存
df_sections = pd.DataFrame(section_rows)
df_sections.to_parquet(_helpers.SECTIONS_PARQUET)
print('saved:', _helpers.SECTIONS_PARQUET, 'rows:', len(df_sections))
df_sections.groupby('section_key').size()


## Cell 8: FinBERT トークナイザでチャンク化

FinBERT (`yiyanghkust/finbert-tone`) は金融テキスト特化の BERT 派生モデル。
**入力上限は 512 トークン**（CLS / SEP の特殊トークン込み）。
この notebook ではセンチメント推論は行わず、tokenizer だけ使って
section テキストを 510 トークンのスライディングウィンドウに分割する。
推論は `02_finbert_sentiment.ipynb` で行う。

In [ ]:
# Cell 8a: FinBERT トークナイザのロード
# from_pretrained は HF_HOME (notebook/FILING_NLP/data/hf_cache) を見て
# キャッシュがあればそれを、無ければ HuggingFace Hub から DL する。
# tokenizer は数 KB のみ。重い model (~440MB) のロードはここでは不要。
from transformers import AutoTokenizer

FINBERT_MODEL_ID = 'yiyanghkust/finbert-tone'
tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL_ID)
print('tokenizer:', type(tokenizer).__name__)
print('vocab_size:', tokenizer.vocab_size)
print('model_max_length:', tokenizer.model_max_length)
print('special tokens:', tokenizer.special_tokens_map)


In [ ]:
# Cell 8b: スライディングウィンドウ・チャンク化関数
# 設計:
#   - max_tokens=510: 512 上限 - 2 (CLS + SEP) = 510 を 1 チャンクに収める
#   - stride=128: 隣接チャンク間で 128 トークン重複させ、段落境界で文脈が
#     途切れるのを緩和する (Risk Factors のような長文での hallucination を抑制)
#
# tokenizer の使い方:
#   - tokenizer(text, add_special_tokens=False) で CLS/SEP 抜きの input_ids
#   - tokenizer.decode(ids, skip_special_tokens=True) で再構築 (チャンク文字列を得る)
#   - decode の境界はトークン単位なので、サブワードが切れることはない
from tqdm.auto import tqdm

MAX_TOKENS = 510
STRIDE = 128

def chunk_text(text: str, tokenizer, max_tokens: int = MAX_TOKENS, stride: int = STRIDE) -> list[str]:
    if not text or not text.strip():
        return []
    token_ids = tokenizer(
        text, add_special_tokens=False, truncation=False,
        return_attention_mask=False,
    )['input_ids']
    if len(token_ids) <= max_tokens:
        return [text]
    step = max_tokens - stride  # 各チャンクで新規に進めるトークン数
    if step <= 0:
        raise ValueError(f'max_tokens ({max_tokens}) must exceed stride ({stride})')
    chunks = []
    start = 0
    while start < len(token_ids):
        window = token_ids[start : start + max_tokens]
        chunks.append(tokenizer.decode(window, skip_special_tokens=True))
        if start + max_tokens >= len(token_ids):
            break
        start += step
    return chunks

# 動作確認: df_sections の 1 件目で chunk 数と先頭の token 数を表示
sample = df_sections.iloc[0]
sample_chunks = chunk_text(sample['text'], tokenizer)
print(f"sample {sample['filing_id']} / {sample['section_key']} "
      f"({sample['char_count']} chars) -> {len(sample_chunks)} chunks")
for i, c in enumerate(sample_chunks[:3]):
    n_tok = len(tokenizer.encode(c, add_special_tokens=False))
    print(f'  chunk[{i}] {n_tok} tokens | head: {c[:80]!r}')


In [ ]:
# Cell 8c: 全 section に chunk_text を適用 → chunk_rows を構築
chunk_rows = []
for row in tqdm(df_sections.to_dict('records'), desc='chunking'):
    chunks = chunk_text(row['text'], tokenizer)
    for i, c in enumerate(chunks):
        chunk_rows.append({
            'filing_id': row['filing_id'],
            'section_key': row['section_key'],
            'chunk_idx': i,
            'text': c,
            'token_count': len(tokenizer.encode(c, add_special_tokens=False)),
        })
print('total chunks:', len(chunk_rows))


In [ ]:
# Cell 9: chunks.parquet 保存 + ticker/form 結合
df_chunks = pd.DataFrame(chunk_rows).merge(
    df_filings[['filing_id', 'ticker', 'form', 'filing_date']],
    on='filing_id', how='left',
)
df_chunks.to_parquet(_helpers.CHUNKS_PARQUET)
print('saved:', _helpers.CHUNKS_PARQUET, 'rows:', len(df_chunks))
df_chunks.groupby(['ticker', 'form', 'section_key']).size()


## Cell 10: 企業名エンティティ抽出 (NER)

FinBERT (finbert-tone) は sequence classification のみで NER 非対応のため、
ここでは **`dslim/bert-base-NER`** を使う。これは CoNLL-2003 で fine-tune
された BERT で、4 ラベル (**ORG / PER / LOC / MISC**) の **token-level**
分類モデル (`AutoModelForTokenClassification`)。サイズ ~110MB。

10-K/10-Q から **ORG (企業名・組織名)** を抽出する。入力は `chunks.parquet`
の chunk (510 トークン以下に分割済み) を使うことで BERT の 512 トークン
上限に確実に収める。

In [ ]:
# Cell 10a: NER モデルのロード (pipeline 経由)
# transformers.pipeline は AutoTokenizer + AutoModelForTokenClassification を
# 内部で組み立てる。aggregation_strategy='simple' でサブワード ('Goog' + '##le')
# を 1 word ('Google') にまとめてくれる。
#
# device 指定: pipeline は torch device の番号 (int) または 'mps' 文字列を
# 取る。get_device() の戻り値が torch.device('mps') の場合は str() で 'mps' に。
from transformers import pipeline

NER_MODEL_ID = 'dslim/bert-base-NER'
ner = pipeline(
    'ner', model=NER_MODEL_ID, tokenizer=NER_MODEL_ID,
    aggregation_strategy='simple',
    device=str(device) if str(device) == 'mps' else -1,
)
print('NER model:', NER_MODEL_ID)
print('id2label:', ner.model.config.id2label)


In [ ]:
# Cell 10b: サンプル 1 chunk で動作確認 (ORG エンティティを表示)
# pipeline 出力: list[dict] で、各 dict は
#   {'entity_group': 'ORG', 'score': 0.99, 'word': 'Apple Inc.',
#    'start': 12, 'end': 22}
# のような形式。start/end は元テキストの文字オフセット。
sample_chunk = df_chunks.iloc[0]
sample_ents = ner(sample_chunk['text'])
orgs = [e for e in sample_ents if e['entity_group'] == 'ORG']
print(f"sample {sample_chunk['filing_id']} / {sample_chunk['section_key']} "
      f"chunk[{sample_chunk['chunk_idx']}] → {len(orgs)} ORG entities")
for e in orgs[:10]:
    print(f"  {e['word']!r} score={e['score']:.3f} pos=[{e['start']}:{e['end']}]")


In [ ]:
# Cell 10c: 全 chunks に NER を適用 → ORG だけ entities.parquet に保存
# pipeline はリスト入力でバッチ推論できる。batch_size で内部マイクロバッチを制御。
# tqdm で進捗を見せるため chunk 単位ループにする (バッチ間で進捗 update)。
BATCH_SIZE = 16
entity_rows = []
texts = df_chunks['text'].tolist()
meta = df_chunks[['filing_id','ticker','form','section_key','chunk_idx']].to_dict('records')

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc='ner'):
    batch_texts = texts[start:start+BATCH_SIZE]
    # pipeline はリストを受けると list[list[dict]] を返す
    batch_results = ner(batch_texts)
    for i, ents in enumerate(batch_results):
        m = meta[start + i]
        for e in ents:
            if e['entity_group'] != 'ORG':
                continue
            entity_rows.append({
                'filing_id': m['filing_id'],
                'ticker': m['ticker'],
                'form': m['form'],
                'section_key': m['section_key'],
                'chunk_idx': m['chunk_idx'],
                'org': e['word'].strip(),
                'score': float(e['score']),
                'start': int(e['start']),
                'end': int(e['end']),
            })

df_entities = pd.DataFrame(entity_rows)
df_entities.to_parquet(_helpers.ENTITIES_PARQUET)
print('saved:', _helpers.ENTITIES_PARQUET, 'rows:', len(df_entities))
# 出現頻度 top 20
df_entities['org'].str.lower().value_counts().head(20)
